In [ ]:
# The following script assumes all Lung Cancer Data from Tisch is downloaded into the same folder as they can all be processed in the same way
import scanpy as sc
import pandas as pd
import anndata as ad
import os
from pathlib import Path

# --- CONFIGURATION ---
dataset_list = [
    'NSCLC_GSE148071',
    'NSCLC_GSE150660', 
    'NSCLC_GSE117570', 
    'NSCLC_GSE127465', 
    'NSCLC_GSE143423',
    'SCLC_GSE150766'
]

# Base folder where all your TISCH downloads live
base_dir = Path(r"path_to_Lung_Tisch_Downloads")

# --- FUNCTIONS ---
def get_label(val):
    """
    Standardizes labels across datasets.
    Adjust keywords if specific datasets use different terms (e.g. 'Cancer' vs 'Malignant')
    """
    val = str(val).lower()
    
    # Class 1: Malignant / Tumor
    if 'malignant' in val and 'non' not in val: 
        return 1
    
    # Class 0: Non-malignant / Normal / Immune / Others
    if 'normal' in val or 'immune' in val or 'non-malignant' in val or 'others' in val:
        return 0
        
    return -1 # Ambiguous / Exclude

# --- MAIN LOOP ---
for dataset in dataset_list:
    print(f"\n{'='*10} Processing: {dataset} {'='*10}")
    
    # 1. Dynamic Paths
    h5_path = base_dir / f"{dataset}_expression.h5"
    meta_path = base_dir / f"{dataset}_CellMetainfo_table.tsv"
    save_path = base_dir / f"{dataset}_annotated.h5ad"
    
    # Skip if files missing
    if not h5_path.exists() or not meta_path.exists():
        print(f"❌ Missing files for {dataset}. Skipping.")
        continue

    try:
        # 2. LOAD METADATA
        print("   Loading Metadata...")
        meta_df = pd.read_csv(meta_path, sep='\t', index_col=0)
        print(f"   Metadata Shape: {meta_df.shape}")

        # 3. LOAD DATA (EXPRESSION MATRIX)
        print("   Loading Expression Matrix...")
        try:
            adata = sc.read_h5ad(h5_path)
        except:
            # The Critical Fix: gex_only=False
            print("   Trying 10x H5 format (gex_only=False)...")
            adata = sc.read_10x_h5(h5_path, gex_only=False)
            adata.var_names_make_unique()

        print(f"   Matrix Shape: {adata.shape}") 

        # 4. ALIGN IDS
        common_cells = adata.obs_names.intersection(meta_df.index)
        print(f"   Overlapping Cells: {len(common_cells)}")

        if len(common_cells) == 0:
            print(f"⚠️  WARNING: 0 overlap for {dataset}. Skipping (IDs might need fixing).")
            continue
        
        # Filter to overlap
        adata = adata[common_cells].copy()
        adata.obs = meta_df.loc[common_cells]

        # 5. CREATE LABELS
        if 'Celltype (malignancy)' in adata.obs.columns:
            adata.obs['da_label'] = adata.obs['Celltype (malignancy)'].map(get_label)
            
            # Print stats
            counts = adata.obs['da_label'].value_counts()
            print(f"   Final Labels: {counts.to_dict()}")
            
            # 6. SAVE
            if len(adata) > 0:
                adata.write_h5ad(save_path, compression='gzip')
                print(f"✅  Success! Saved to {save_path.name}")
            else:
                print("⚠️  No labeled cells left after filtering.")
                
        else:
            print(f"❌  Column 'Celltype (malignancy)' not found in metadata for {dataset}.")

    except Exception as e:
        print(f"❌  Error processing {dataset}: {e}")

print("\n--- Batch Processing Complete ---")


========== Processing: BRCA_GSE161529 ==========
   Loading Metadata...
   Metadata Shape: (332168, 9)
   Loading Expression Matrix...
   Trying 10x H5 format (gex_only=False)...
   Matrix Shape: (332168, 27188)
   Overlapping Cells: 332168
   Final Labels: {1: 151781, -1: 124097, 0: 56290}
✅  Success! Saved to BRCA_GSE161529_annotated.h5ad

--- Batch Processing Complete ---
